# YOLOv8s — Small Model + Oversample + Optimized Augmentation

**Tại sao dùng YOLOv8s (Small)?**
- Kiến trúc lớn hơn YOLOv8n (~3.5x tham số), giúp học được pattern phức tạp hơn
- Kết hợp Oversample + Augmentation vừa phải mà không bị underfitting
- Inference vẫn đủ nhanh cho ứng dụng thực tế

**Pipeline:**
1. Clone repo + Tải dữ liệu từ Google Drive
2. Cài đặt thư viện
3. Làm sạch → Chuyển COCO→YOLO → Chia tập
4. Oversample class thiểu số
5. Huấn luyện YOLOv8s (100 epoch, optimized aug)
6. Đánh giá trên tập Test
7. Xuất kết quả

## 1. Clone repo & Tải dữ liệu

In [ ]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

In [ ]:
!pip install gdown
# Tải file từ Drive về Kaggle
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip

# Tạo thư mục data
!mkdir -p /kaggle/working/waste-detection_project/data

# Giải nén vào thư mục data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/

# Dọn dẹp file zip để tránh bị đầy bộ nhớ (Disk quota exceeded)
!rm raw.zip

## 2. Cài đặt thư viện

In [ ]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt

## 3. Xử lý dữ liệu: Làm sạch → Chuyển đổi → Chia tập

In [ ]:
# Làm sạch dữ liệu
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

# Chuyển COCO sang YOLO format
%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

# Chia tập train/val/test
%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

## 4. Oversample class thiểu số (Glass, Paper, Metal)

Chỉ oversample, **KHÔNG** chạy letterbox thủ công (YOLO tự xử lý padding nội bộ).

In [ ]:
%cd /kaggle/working/waste-detection_project/src/data_prep
!python oversample.py

## 5. Sửa đường dẫn dataset.yaml cho Kaggle

In [ ]:
# Quay về thư mục experimental trước khi train
%cd /kaggle/working/waste-detection_project/notebooks/experimental

yaml_file = '/kaggle/working/waste-detection_project/data/processed/dataset.yaml'

with open(yaml_file, 'r') as f:
    content = f.read()

# Thay thế đường dẫn tương đối bằng đường dẫn tuyệt đối trên Kaggle
import re
content = re.sub(
    r'path:.*',
    'path: /kaggle/working/waste-detection_project/data/processed',
    content
)

with open(yaml_file, 'w') as f:
    f.write(content)

print("Da sua duong dan file dataset.yaml thanh cong!")
print()
with open(yaml_file, 'r') as f:
    print(f.read())

## 6. Huấn luyện YOLOv8s (Optimized)

**Chiến lược:**
- `yolov8s.pt`: Model Small (~11.2M params, gấp ~3.5x nano)
- `epochs=100` + `patience=20`: Train dài hơn với early stopping
- `cos_lr=True`: Cosine Annealing LR scheduler
- `batch=16`: Giữ nguyên batch size (YOLOv8s vẫn chạy tốt với bs=16 trên GPU Kaggle)
- Augmentation vừa phải: mosaic 0.8, mixup 0.05 (model lớn có thể xử lý được), scale 0.4
- `close_mosaic=15`: Tắt mosaic 15 epoch cuối

In [ ]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path('/kaggle/working/waste-detection_project')
YAML_PATH = BASE_DIR / "data/processed/dataset.yaml"
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

print("=" * 55)
print("  YOLOv8s — 100 Epoch + Optimized Aug + Oversample")
print("=" * 55)

model_path = BASE_DIR / "models/yolov8s.pt"
model = YOLO(str(model_path))

train_results = model.train(
    data        = str(YAML_PATH),
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    name        = "yolov8s_improved",
    project     = str(RESULTS / "runs"),
    exist_ok    = True,
    verbose     = True,
    # --- Optimized Hyperparameters ---
    patience    = 20,         # Early stopping: dừng nếu 20 epoch không cải thiện
    cos_lr      = True,       # Cosine Annealing LR
    close_mosaic = 15,        # Tắt mosaic 15 epoch cuối để fine-tune
    # --- Augmentation (model lớn hơn => chịu được aug mạnh hơn) ---
    degrees     = 5.0,        # Xoay nhẹ
    translate   = 0.1,        # Dịch chuyển 10%
    scale       = 0.4,        # Thu phóng (model lớn xử lý tốt hơn nano)
    fliplr      = 0.5,        # Lật ngang 50%
    mosaic      = 0.8,        # Mosaic 80%
    mixup       = 0.05,       # Mixup nhẹ (model lớn chịu được)
)

print("\n" + "=" * 55)
print("  HUAN LUYEN HOAN TAT!")
print("=" * 55)

## 7. Đánh giá trên tập Test

In [ ]:
# Đánh giá trên tập Test
print("\n[EVAL] Danh gia tren tap Test ...")
metrics = model.val(split="test", verbose=True)

map50   = float(metrics.box.map50)
map5095 = float(metrics.box.map)
prec    = float(metrics.box.mp)
recall  = float(metrics.box.mr)

print(f"\n{'='*40}")
print(f"  KET QUA TREN TAP TEST")
print(f"{'='*40}")
print(f"  mAP@0.5     : {map50:.4f} ({map50*100:.2f}%)")
print(f"  mAP@0.5:0.95: {map5095:.4f} ({map5095*100:.2f}%)")
print(f"  Precision   : {prec:.4f} ({prec*100:.2f}%)")
print(f"  Recall      : {recall:.4f} ({recall*100:.2f}%)")
print(f"{'='*40}")

# Đo tốc độ inference
val_imgs = glob.glob(str(BASE_DIR / "data/processed/images/test/**/*.*"), recursive=True)[:50]
if val_imgs:
    t0 = time.perf_counter()
    for img_path in val_imgs:
        model.predict(img_path, verbose=False)
    elapsed = (time.perf_counter() - t0) / len(val_imgs) * 1000
    print(f"  Inference   : {elapsed:.1f} ms/anh")
else:
    elapsed = None
    print("  [WARN] Khong tim thay anh test de do inference.")

## 8. Lưu kết quả & Xuất file ZIP

In [ ]:
# Lưu kết quả vào CSV
result_row = {
    "Model": "YOLOv8s Improved (100ep, opt-aug, oversample)",
    "mAP@0.5 (%)": round(map50 * 100, 2),
    "mAP@0.5:0.95 (%)": round(map5095 * 100, 2),
    "Precision (%)": round(prec * 100, 2),
    "Recall (%)": round(recall * 100, 2),
    "Inference (ms)": round(elapsed, 1) if elapsed else "N/A",
}
df = pd.DataFrame([result_row])
csv_path = RESULTS / "ket_qua_yolov8s_improved.csv"
df.to_csv(csv_path, index=False)
print(f"\nDa luu ket qua tai: {csv_path}")
print(df.to_string(index=False))

In [ ]:
# Nén kết quả để tải về
!zip -r /kaggle/working/yolov8s_improved_results.zip /kaggle/working/waste-detection_project/results/runs/yolov8s_improved
print("\nFile ZIP da san sang de tai ve!")